TA Version

In [1]:
!pip install -q transformers peft accelerate sentencepiece huggingface_hub

Imports

In [2]:
import json
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

download the sample dataset from Hugging Face

In [3]:
DATASET_REPO = "YTJJGaming/spider-sample"

dev_path = hf_hub_download(
    repo_id=DATASET_REPO,
    filename="dev_sample.json",
    repo_type="dataset"
)

tables_path = hf_hub_download(
    repo_id=DATASET_REPO,
    filename="tables.json",
    repo_type="dataset"
)

with open(dev_path, "r", encoding="utf-8") as f:
    dev_examples = json.load(f)

with open(tables_path, "r", encoding="utf-8") as f:
    tables = json.load(f)

db_index = {item["db_id"]: item for item in tables}

print("Loaded sample examples:", len(dev_examples))
print("Loaded schemas:", len(tables))
print("First question:", dev_examples[0]["question"])

dev_sample.json: 0.00B [00:00, ?B/s]

c:\Users\jesse\Desktop\CS175\CS-175-Project\venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jesse\.cache\huggingface\hub\datasets--YTJJGaming--spider-sample. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tables.json: 0.00B [00:00, ?B/s]

Loaded sample examples: 10
Loaded schemas: 166
First question: How many singers do we have?


helper to turn one Spider schema into text

In [4]:
def simple_serialize_schema(db_schema):
    table_names = db_schema["table_names_original"]
    column_names = db_schema["column_names_original"]

    table_to_columns = {i: [] for i in range(len(table_names))}
    for table_id, col_name in column_names:
        if table_id == -1:
            continue
        table_to_columns[table_id].append(col_name)

    lines = []
    for i, table_name in enumerate(table_names):
        cols = ", ".join(table_to_columns[i])
        lines.append(f"Table {table_name}({cols})")

    return "\n".join(lines)

pick one example and show the prompt pieces

In [5]:
example = dev_examples[0]
question = example["question"]
gold_sql = example["query"]
db_id = example["db_id"]
schema_text = simple_serialize_schema(db_index[db_id])

print("DB ID:", db_id)
print("\nQuestion:")
print(question)
print("\nGold SQL:")
print(gold_sql)
print("\nSchema:")
print(schema_text[:1500])

DB ID: concert_singer

Question:
How many singers do we have?

Gold SQL:
SELECT count(*) FROM singer

Schema:
Table stadium(Stadium_ID, Location, Name, Capacity, Highest, Lowest, Average)
Table singer(Singer_ID, Name, Country, Song_Name, Song_release_year, Age, Is_male)
Table concert(concert_ID, concert_Name, Theme, Stadium_ID, Year)
Table singer_in_concert(concert_ID, Singer_ID)


download the model from Hugging Face

In [6]:
BASE_MODEL = "google/gemma-3-4b-it"
ADAPTER_REPO = "YTJJGaming/gemma-text2sql"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model.eval()

print("Model loaded.")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Model loaded.


Demo

In [7]:
prompt = f"""You are a Text-to-SQL model.

Schema:
{schema_text}

Question: {question}
Return only SQL.
"""

inputs = tokenizer(prompt, return_tensors="pt")

if torch.cuda.is_available():
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

prompt_len = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][prompt_len:]
pred_sql = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("Question:")
print(question)
print("\nPredicted SQL:")
print(pred_sql)
print("\nGold SQL:")
print(gold_sql)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question:
How many singers do we have?

Predicted SQL:


Gold SQL:
SELECT count(*) FROM singer
